# 41 - Humanoid Edge Compute

## What / Why / How

**What we are trying to do:** Estimate compute budgets for humanoid perception, planning, control, and VLA inference.

**Why this matters:** A useful humanoid must react in real time. Large models must coexist with high-frequency control loops.

**How we will do it:** Create latency budgets, compare synchronous versus asynchronous inference, and simulate missed control deadlines.

## Edge Compute

A humanoid has fast loops and slow loops. Joint control might run at 500-2000 Hz, locomotion at 50-200 Hz, perception at 10-60 Hz, and high-level VLA reasoning slower.

In [ ]:
import math
import random
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

try:
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ModuleNotFoundError:
    plt = None
    HAS_PLOT = False

np.set_printoptions(precision=3, suppress=True)

def plot_unavailable():
    if not HAS_PLOT:
        print("Install matplotlib to see plots: pip install -r requirements.txt")

In [ ]:
loops = {
    "joint_current": {"hz": 1000, "budget_ms": 1.0},
    "whole_body_control": {"hz": 200, "budget_ms": 5.0},
    "locomotion_policy": {"hz": 50, "budget_ms": 20.0},
    "vision_tracking": {"hz": 30, "budget_ms": 33.3},
    "vla_action_chunk": {"hz": 5, "budget_ms": 200.0},
    "task_planner": {"hz": 1, "budget_ms": 1000.0},
}
for name, cfg in loops.items():
    print(f"{name:20} {cfg['hz']:4} Hz budget={cfg['budget_ms']:6.1f} ms")

In [ ]:
rng = np.random.default_rng(41)
measured_latency = {
    "whole_body_control": rng.normal(3.2, 0.8, 200),
    "vision_tracking": rng.normal(24, 8, 200),
    "vla_action_chunk": rng.normal(130, 45, 200),
}
for name, values in measured_latency.items():
    budget = loops[name]["budget_ms"]
    miss_rate = np.mean(values > budget)
    print(name, "p95", np.percentile(values, 95), "miss_rate", miss_rate)

In [ ]:
# Async inference: fast controller reuses the last action chunk while slow VLA computes the next one.
control_steps = 120
chunk_period = 10
current_chunk_id = 0
used_chunks = []
for step in range(control_steps):
    if step % chunk_period == 0:
        current_chunk_id += 1
    used_chunks.append(current_chunk_id)
print("chunks used:", used_chunks[:25], "...", used_chunks[-5:])

## Exercises

1. Add camera preprocessing latency.
2. Add a watchdog if VLA outputs stop arriving.
3. Explain which loops must never depend directly on cloud inference.